# 04. 내구성 실행 (Durable Execution)

> 프로덕션 에이전트는 도중에 죽어도 다시 살아나야 해요. 체크포인터 기반 복원, `RetryPolicy`, 멱등성 키로 내구성 있는 실행 계층을 만들어요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 내구성 실행(Durable Execution)의 3가지 핵심 요소를 설명하고 직접 구성할 수 있어요
2. 3가지 Durability Mode(exit, async, sync)의 차이를 이해하고 상황에 맞게 선택할 수 있어요
3. 장애 상황을 시뮬레이션하고 같은 thread_id로 그래프 실행을 재개할 수 있어요
4. RetryPolicy를 설정하여 일시적인 오류에 자동으로 재시도하는 그래프를 만들 수 있어요
5. @task + @entrypoint 패턴으로 사이드 이펙트를 격리하는 Functional API를 작성할 수 있어요

## 사전 지식

- `03-Branching-Parallel.ipynb`: Conditional Edges, 병렬 실행, Fan-out/Fan-in 패턴
- Part 2에서 배운 Checkpointer(InMemorySaver, SqliteSaver)와 thread_id 개념
- StateGraph의 노드·엣지 구성 방식


## 내구성 실행이란?

에이전트 워크플로우를 프로덕션에서 실행하다 보면 반드시 **중간에 실패하는 경우**가 생겨요.
네트워크 오류, API 속도 제한(Rate Limit), 서버 재시작 등 다양한 이유로 그래프 실행이 중단될 수 있죠.

**내구성 실행(Durable Execution)**은 이런 상황에서도 **처음부터 다시 시작하지 않고**, 중단된 지점에서 이어갈 수 있게 해주는 기능이에요.

### 왜 필요한가요?

온라인 쇼핑을 하다가 결제 직전에 인터넷이 끊겼다고 상상해보세요. 다시 접속했을 때 장바구니가 비어있으면 처음부터 다시 담아야 해요. 하지만 장바구니가 **저장**되어 있으면 결제만 다시 하면 되죠. 내구성 실행이 바로 이런 역할을 해요.

> 🔑 **핵심 개념**: 내구성 실행의 3가지 필수 요소
> 1. **Checkpointer**: 각 스텝의 상태를 저장하는 저장소 (InMemorySaver, SqliteSaver, PostgresSaver)
> 2. **thread_id**: 동일한 실행 흐름을 식별하는 고유 ID
> 3. **결정론성(Determinism)**: 재개 시 동일한 경로로 흐르도록 랜덤/사이드 이펙트를 @task 안에 격리

```mermaid
flowchart TD
    A["그래프 실행 시작<br/>thread_id: abc-123"] --> B["Node 1 실행<br/>체크포인트 저장"]
    B --> C["Node 2 실행<br/>체크포인트 저장"]
    C --> D{"Node 3 실행 중<br/>장애 발생!"}
    D --> E["graph.invoke(None, config)<br/>같은 thread_id로 재개"]
    E --> F["Node 3부터 재시작<br/>(Node 1, 2는 건너뜀)"]
    F --> G["Node 4 실행<br/>완료"]

    classDef ok fill:#d4edda,stroke:#28a745,color:#155724
    classDef fail fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef resume fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A,B,C ok
    class D fail
    class E,F resume
    class G ok
```

### Durability Mode 비교

| Mode | 체크포인트 생성 시점 | 사용 상황 | 비유 |
|------|------------------|-----------|------|
| `"exit"` (기본값) | 노드 완료 시 | 일반적인 사용, 대부분의 경우 적합 | 각 역 도착 시 기록 |
| `"async"` | 비동기 작업 완료 시 | 오래 걸리는 비동기 작업이 있을 때 | 비동기 작업마다 기록 |
| `"sync"` | 모든 스텝마다 | 최대 내구성 필요, 중단점이 많을수록 재개가 세밀해짐 | 매 발걸음마다 기록 |

### Super-step과 트랜잭션 보장

> 🔑 **핵심 개념**: Super-step은 체크포인트 생성 단위예요.
> 하나의 Super-step 안에서 여러 노드가 실행될 수 있어요.
> 만약 Super-step 중간에 실패하면 **해당 Super-step 전체가 롤백**되어, 이전 체크포인트에서 재개돼요.
> 이는 데이터베이스 트랜잭션과 유사한 "all-or-nothing" 보장을 제공해요.


## 환경 설정


In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택 사항)
# ---------------------------------------------------
# 실행 과정을 LangSmith에서 시각적으로 확인할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-04-Durable-Execution"


## 1. 기본 내구성 실행 - 장애 시뮬레이션과 재개

먼저 가장 기본적인 내구성 실행 패턴을 살펴볼게요.
세 개의 노드로 구성된 간단한 그래프에서 두 번째 노드가 실패하는 상황을 만들고,
같은 `thread_id`로 재개해볼 거예요.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 내구성 실행 그래프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import uuid

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 1차 실행: 의도적으로 Step 2에서 실패시키기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 체크포인트 상태 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 2차 실행: None 입력으로 중단 지점에서 재개
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### Checkpointer 종류와 선택 기준

| Checkpointer | 저장 위치 | 사용 환경 | 설치 |
|-------------|---------|---------|------|
| `InMemorySaver` | 메모리 | 개발/테스트 | 기본 포함 |
| `SqliteSaver` | SQLite 파일 | 로컬/소규모 | `langgraph-checkpoint-sqlite` |
| `PostgresSaver` | PostgreSQL DB | 프로덕션 | `langgraph-checkpoint-postgres` |


## 2. RetryPolicy - 자동 재시도

### 왜 수동 재개 대신 자동 재시도가 필요한가요?

섹션 1에서는 장애 발생 후 수동으로 `graph.invoke(None, config)`를 호출해서 재개했어요. 하지만 API 타임아웃처럼 **잠깐 기다리면 자동으로 해결되는 오류**에 매번 사람이 개입하는 것은 비효율적이에요. 마치 전화가 안 터질 때 자동으로 재다이얼하는 것처럼, LangGraph의 **RetryPolicy**를 사용하면 일시적인 오류에 자동으로 재시도할 수 있어요.


In [ ]:
from langgraph.types import RetryPolicy

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: RetryPolicy 설정 예시
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import random

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: RetryPolicy가 적용된 그래프 구성
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → api_call → process → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: RetryPolicy 동작 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 결정론성과 @task 패턴

### 왜 비교 예제가 필요한가요?

내구성 실행에서 가장 헷갈리는 지점은 **"재개(resume)는 멈춘 코드 줄에서 그대로 이어지는가?"**예요.  
정답은 **아니에요**. `interrupt()` 뒤에 `Command(resume=...)`로 재개하면 LangGraph는 체크포인트를 기준으로 실행을 다시 구성하고, Functional API에서는 중단된 `@entrypoint`의 **처음부터 replay**해요.

그래서 `interrupt()` 앞에 일반 Python 코드로 API 호출, 랜덤값 생성, 파일 쓰기 같은 사이드 이펙트를 넣으면 재개 과정에서 그 코드가 다시 실행될 수 있어요.

이번 섹션은 같은 흐름을 두 번 비교합니다.

1. ❌ **비교 대상**: `@task` 없이 API 호출을 직접 실행
2. ✅ **권장 패턴**: 같은 API 호출을 `@task`로 감싸서 결과를 체크포인트에 저장

| 패턴 | `Command(resume=...)` 시 동작 | 결과 |
|------|-----------------------------|------|
| `@entrypoint` 안에서 직접 API 호출 | `interrupt()` 이전 코드가 replay되며 API가 다시 호출됨 | 중복 호출/다른 결과 위험 |
| API 호출을 `@task`로 감싸기 | 완료된 task 결과를 체크포인트에서 읽음 | 중복 호출 방지, 같은 결과 유지 |

> 🔑 **핵심 개념**: `@task`는 "오래 걸리거나, 외부 세계를 바꾸거나, 매번 값이 달라질 수 있는 작업"을 replay-safe하게 만드는 단위예요.
>
> - 외부 API 호출
> - 파일 쓰기 / 이메일 발송
> - 랜덤값, 현재 시각, UUID 생성
> - 비용이 큰 LLM 호출 또는 데이터 처리

> ⚠️ **주의**: 같은 `thread_id`로 **정상 완료된 워크플로우를 다시 `invoke()`**하는 것은 resume이 아니에요.  
> 이 섹션의 비교는 `interrupt()`로 멈춘 뒤 **같은 `thread_id` + `Command(resume=...)`**로 재개하는 상황을 기준으로 봐야 해요.


### 3-0. 실험을 읽는 방법

아래 3-1과 3-2는 **같은 사용자 데이터 조회 흐름**을 두 가지 방식으로 실행해 보는 비교 실험이에요.

- **3-1**: API 호출을 일반 Python 함수로 직접 실행합니다. `resume` 때 같은 함수가 다시 실행되는 문제가 보입니다.
- **3-2**: API 호출을 `@task`로 감쌉니다. `resume` 때 완료된 task 결과를 체크포인트에서 재사용합니다.

> 🧭 **권장 실행 순서**  
> 3-1을 끝까지 실행해서 **API가 2번 호출되는 문제**를 확인한 뒤, 3-2를 실행해서 **API가 1번만 호출되는 차이**를 확인하세요.


In [ ]:
# ---------------------------------------------------
# 3-0. 공통 준비
# ---------------------------------------------------
# 두 실험 모두 같은 LangGraph Functional API 도구를 사용해요.
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt
from pprint import pprint
import uuid


### 3-1. 비교 대상: `@task` 없이 직접 사이드 이펙트 실행

먼저 일부러 좋지 않은 패턴을 봅니다.  
`interrupt()` 전에 외부 API 같은 작업을 **그냥 함수로 직접 호출**하면, `Command(resume=...)`로 재개할 때 entrypoint가 replay되면서 그 함수도 다시 실행될 수 있어요.

이번 예제에서는 API가 호출될 때마다 `api_call_no`와 `score`가 달라지도록 만들어서 중복 호출을 눈으로 확인합니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-1-A. 직접 호출 방식: API 함수와 관찰용 로그 준비
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-1-B. 나쁜 예: entrypoint 안에서 API를 직접 호출
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-1-C. 첫 실행: interrupt에서 멈추는지 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


#### 3-1 관찰 포인트

위 셀에서 이미 `direct_fetch_user_data()`가 한 번 호출되었습니다.  
이제 같은 `thread_id`로 `Command(resume=...)`를 보내면, `interrupt()` 뒤에서 단순히 이어지는 것처럼 보이지만 실제로는 entrypoint가 replay됩니다.

따라서 아래 셀을 실행하면 `[Direct API] 사용자 데이터 호출 #2`가 찍히는지 확인하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-1-D. resume: 직접 호출 함수가 다시 실행되는 문제 확인
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


#### 3-1-E. 한눈에 보기: 직접 호출은 다시 실행돼요

```mermaid
flowchart LR
    A[첫 실행] --> B[API 직접 호출 #1]
    B --> C[interrupt에서 멈춤]
    C --> D[resume]
    D --> E[처음부터 replay]
    E --> F[API 직접 호출 #2]
    F --> G[결과: API 총 2회 호출]
```

> **핵심**: `interrupt()` 전에 일반 함수로 직접 실행한 코드는 `resume` 때 다시 실행될 수 있어요.


### 3-2. 권장 패턴: 사이드 이펙트를 `@task`로 격리

이제 같은 사용자 데이터 조회를 `@task`로 감싸 봅니다.

핵심은 `fetch_user_data(user_id).result()`예요.  
entrypoint는 resume 과정에서 다시 지나가지만, 이미 완료된 `@task` 결과는 체크포인트에 저장되어 있으므로 실제 API 함수 본문은 다시 실행되지 않습니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-2-A. @task 방식: API task와 분석 task 준비
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-2-B. 좋은 예: entrypoint 안에서는 task 결과를 사용
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-2-C. 첫 실행: fetch_user_data task가 1번 실행되고 interrupt에서 멈춤
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


#### 3-2 관찰 포인트

아래 resume 셀을 실행하면 workflow 시작 로그는 다시 찍힙니다.  
하지만 `[Task] 사용자 데이터 호출 #2`는 찍히지 않아야 해요.

대신 interrupt 이후 단계인 `analyze_data()`만 새로 실행됩니다.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-2-D. resume: 완료된 fetch_user_data task 결과를 재사용
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


#### 3-2-E. 한눈에 보기: `@task` 결과는 재사용돼요

```mermaid
flowchart LR
    A[첫 실행] --> B[API task 실행 #1]
    B --> C[결과를 checkpoint에 저장]
    C --> D[interrupt에서 멈춤]
    D --> E[resume]
    E --> F[저장된 task 결과 재사용]
    F --> G[결과: API 총 1회 호출]
```

> **핵심**: 반복 실행되면 안 되는 작업은 `@task`로 감싸면, `resume` 때 완료된 결과를 재사용할 수 있어요.


### 3-3. 결과 비교 요약

두 실험의 차이는 **resume 후 API 호출 횟수와 데이터 일관성**에서 바로 보입니다.

| 항목 | 3-1 직접 호출 | 3-2 `@task` 사용 |
|---|---:|---:|
| 첫 실행 시 API 호출 | 1번 | 1번 |
| resume 후 총 API 호출 | 2번 | 1번 |
| pause 때 `api_call_no` | 1 | 1 |
| resume 결과 `api_call_no` | 2 | 1 |
| 해석 | replay 때문에 사이드 이펙트 중복 실행 | 체크포인트된 task 결과 재사용 |

> ✅ 결론: `interrupt()` 전에 실행되는 외부 API 호출, 결제, 이메일 발송, LLM 호출처럼 **반복 실행되면 안 되는 작업은 `@task`로 감싸세요.**


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-3. 두 실험 결과를 한 번에 비교
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 3-4. LangGraph 기본 그래프 시각화 (참고)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. execution_info - 재시도 정보 접근

RetryPolicy로 재시도 중에 현재 시도 횟수나 첫 시도 시각을 알고 싶을 때가 있어요.
LangGraph 1.1.3 이상에서는 `get_store()` 또는 노드 파라미터를 통해
실행 정보(`execution_info`)에 접근할 수 있어요.


In [ ]:
from langchain_core.runnables import RunnableConfig
from datetime import datetime

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: RunnableConfig를 통한 실행 정보 접근
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 멱등성 키(Idempotency Key)

외부 API를 재시도할 때 중복 호출 문제가 생길 수 있어요.
예를 들어 결제 API를 재시도하면 이중 결제가 발생할 수 있죠.

**멱등성 키(Idempotency Key)**를 사용하면 서버가 동일한 요청임을 인식하고
이전 결과를 반환하도록 할 수 있어요.

> 🔑 **핵심 개념**: 멱등성(Idempotency)은 "같은 요청을 여러 번 보내도 결과가 동일하다"는 속성이에요.
> 대부분의 결제 API(Stripe, PayPal 등)와 메시지 큐(SQS 등)가 멱등성 키를 지원해요.


In [ ]:
import hashlib

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 멱등성 키 패턴
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. SqliteSaver - 영구 저장 체크포인터

`InMemorySaver`는 프로세스가 종료되면 모든 체크포인트가 사라져요.
프로세스 재시작 후에도 재개하려면 **파일 기반 또는 DB 기반 체크포인터**가 필요해요.


In [ ]:
import tempfile
import os

from langgraph.checkpoint.sqlite import SqliteSaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: SqliteSaver를 사용한 영구 체크포인트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 7. TODO 실습: 장애 복구 시나리오 설계

아래 코드는 3단계 데이터 파이프라인이에요.
여러분이 직접 RetryPolicy와 재개 로직을 완성해보세요.


In [ ]:
from langgraph.types import RetryPolicy
from langgraph.checkpoint.memory import InMemorySaver

# ============================================================
# TODO: 아래 데이터 파이프라인에 내구성 실행을 추가하세요
#
# 요구사항:
#   1. extract_node: ConnectionError 발생 시 최대 3회 재시도
#      (initial_interval=0.1, backoff_factor=2.0)
#   2. transform_node: 아무 설정 없이 추가 (재시도 없음)
#   3. load_node: TimeoutError 발생 시 최대 2회 재시도
#
# 힌트:
#   - add_node(..., retry=RetryPolicy(...)) 형태로 각 노드에 개별 적용
#   - retry_on 파라미터에 예외 타입 또는 lambda 함수를 전달하세요
#   - graph.compile(checkpointer=InMemorySaver())로 체크포인터 연결
#
# 예상 결과:
#   extract_node가 실패해도 자동 재시도 후 정상 완료
# ============================================================


# TODO: 아래 코드를 완성하세요


# TODO 1: extract_node를 ConnectionError 재시도 RetryPolicy와 함께 추가하세요


# TODO 2: transform_node를 추가하세요 (재시도 없음)


# TODO 3: load_node를 TimeoutError 재시도 RetryPolicy와 함께 추가하세요


# TODO 4: InMemorySaver로 컴파일하세요


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **내구성 실행(Durable Execution)**: Checkpointer + thread_id + 결정론성 세 가지 요소로 구현해요. 장애 발생 시 처음부터 재시작하지 않고 중단점에서 재개해요.
- **Durability Mode**: `"exit"`(기본), `"async"`, `"sync"` 세 가지 모드로 체크포인트 생성 시점을 제어해요.
- **장애 재개 패턴**: `graph.invoke(None, config)`에 동일 `thread_id`를 사용하면 자동으로 마지막 체크포인트에서 이어가요.
- **RetryPolicy**: `max_attempts`, `initial_interval`, `backoff_factor`, `jitter`, `retry_on` 파라미터로 자동 재시도를 구성해요. `ValueError`, `TypeError` 등 프로그래밍 오류는 기본적으로 재시도하지 않아요.
- **@task + @entrypoint**: Functional API에서 사이드 이펙트를 격리하는 패턴이에요. `@task` 결과는 캐시되어 재개 시 재실행되지 않아요.
- **멱등성 키**: thread_id + 노드명 + 연산 조합으로 재현 가능한 키를 만들어 외부 API 중복 호출을 방지해요.
- **Checkpointer 종류**: `InMemorySaver`(개발), `SqliteSaver`(로컬), `PostgresSaver`(프로덕션)을 용도에 맞게 선택해요.


## 다음 노트북 예고

다음 `05-DeleteMessages.ipynb`에서는 **`RemoveMessage`로 컨텍스트 윈도우 최적화**를 배워요. 대화가 길어질수록 토큰 비용과 지연이 늘어나는데, 메시지 ID를 지정해 불필요한 과거 메시지를 정밀하게 삭제하는 방법을 다뤄요. 오늘 배운 체크포인터에 쌓이는 상태를 효과적으로 관리하는 후속 기법이에요.
